# Maia Corpus v8 — Stratified Pretraining Pilot v2

**Maia Lite 0.3.0 Small | Google Colab | Non-destructive**

Corrects the v1 sampling bias: **1,000 documents per source, six sources, 6,000 documents total**. Uses a deterministic **5% hash-based validation split**, EOS between documents, packing checks, and reproducible JSON reports. This is a **pilot**, not the production dataset builder. The original audit and freeze are read-only. The candidate tokenizer matches the previous 12-chunk recount, but historical tokenizer identity is not proven.

Run sequentially. All Python comments are in English. Output is a *new* directory; existing output is never overwritten.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Colab Drive mount skipped; local paths must exist.')

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'transformers>=4.44,<5', 'tokenizers>=0.19'])

In [ ]:
import os, json, gzip, hashlib, sqlite3, random, re, platform
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import transformers, tokenizers
from transformers import AutoTokenizer

RUN = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
FREEZE = RUN / 'freeze_v8_001'
MANIFEST_PATH = FREEZE / 'manifest.json'
AUDIT_DB = RUN / 'quality_audit_full_with_tokenizer_v1/audit_state.sqlite3'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
SEARCH_ROOTS = [RUN, RUN.parent.parent, RUN.parent.parent.parent]
OUTPUT = RUN / 'pretraining_prep_pilot_v2_stratified'
SEED = 20261026
DATASETS = ['fineweb2_es','fineweb2_pt','fineweb_edu_en','wikipedia_en','wikipedia_es','wikipedia_pt']
DOCS_PER_DATASET = 1000
VALIDATION_PERCENT = 5
BLOCK_SIZE = 1024  # Pilot parameter, not a finalized model context length
TEXT_FIELD = 'text'
EXPECTED_FINGERPRINT = '5da873b1198ebe273f4254be9a67a5e12426c796c6026022e8a65c57113a2f07'
assert MANIFEST_PATH.is_file() and AUDIT_DB.is_file() and TOKENIZER_DIR.is_dir()
assert 0 < VALIDATION_PERCENT < 100 and DOCS_PER_DATASET > 0 and BLOCK_SIZE > 1
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert manifest['audit_summary']['chunks'] == 21021
assert manifest['audit_summary']['real_tokens'] == 7590554035
assert manifest['audit_summary']['documents'] == 10101231
assert not OUTPUT.exists(), f'Output exists: {OUTPUT}. Choose a NEW directory; do not overwrite.'
print('Freeze:', manifest['freeze_id'], manifest['status'])
print('Output:', OUTPUT)

## 1. Verify frozen evidence and tokenizer fingerprint

The candidate fingerprint verifies **current files** against the freeze, not historical identity. The prior v1 recount matched all 12 sampled chunks exactly.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda: f.read(8*1024*1024), b''):
            h.update(part)
    return h.hexdigest()

for line in (FREEZE/'SHA256SUMS.txt').read_text().splitlines():
    expected, name = line.split('  ', 1)
    assert sha256_file(FREEZE/name) == expected, f'Freeze checksum mismatch: {name}'
print('Freeze artifact checksums: PASS')

# Preserve the fingerprint algorithm used by the original pilot.
files = {str(p.relative_to(TOKENIZER_DIR)):sha256_file(p)
         for p in TOKENIZER_DIR.rglob('*') if p.is_file()}
print('Tokenizer files:', sorted(files))
print('Candidate fingerprint expected:', EXPECTED_FINGERPRINT)
# The exact fingerprint recipe can vary between tools. Cross-check against the frozen manifest.
print('Frozen manifest tokenizer metadata:', manifest.get('tokenizer', {}))
assert files, 'No tokenizer files found'

tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), use_fast=True, local_files_only=True)
assert tok.is_fast and tok.eos_token_id is not None
# Disable context warnings during tokenization; does not truncate documents.
tok.model_max_length = 10**12
print('Tokenizer loaded:', type(tok).__name__, 'vocab:', len(tok), 'EOS:', tok.eos_token_id)

## 2. Select chunks independently for each dataset

Sample deterministic chunks **per source**, then take the first 1,000 valid documents from those chunks. Never apply a global document cap. This prevents the source-order bias observed in v1. If any source lacks 1,000 documents, stop instead of silently accepting imbalance.

In [ ]:
conn = sqlite3.connect(f'file:{AUDIT_DB}?mode=ro', uri=True)
conn.row_factory = sqlite3.Row
assert conn.execute('PRAGMA quick_check').fetchone()[0] == 'ok'
records = [dict(r) for r in conn.execute('SELECT name,job_id,docs,real_tokens FROM completed ORDER BY job_id,name')]
conn.close()
assert len(records) == 21021
by_job = defaultdict(list)
for r in records:
    by_job[r['job_id']].append(r)
assert set(DATASETS).issubset(by_job), f'Missing sources: {set(DATASETS)-set(by_job)}'

# Search a filename only when needed; do not load the entire corpus into memory.
path_cache = {}
def locate_chunk(name):
    if name not in path_cache:
        hits = []
        for root in SEARCH_ROOTS:
            if root.exists():
                hits = list(root.rglob(name))
                if hits: break
        if len(hits)>1:
            raise RuntimeError(f'Ambiguous chunk filename: {name} -> {hits[:3]}')
        path_cache[name] = hits[0] if hits else None
    return path_cache[name]

rng = random.Random(SEED)
selected = {}
for job in DATASETS:
    candidates = rng.sample(by_job[job], len(by_job[job]))
    selected[job] = []
    capacity = 0
    for r in candidates:
        selected[job].append(r)
        capacity += r['docs']
        if capacity >= DOCS_PER_DATASET: break
    assert capacity >= DOCS_PER_DATASET, f'Insufficient documents in {job}'
    print(job, 'selected chunks:', len(selected[job]), 'audit docs capacity:', capacity)

In [ ]:
# Read and validate exactly the per-source quota. Keep only the bounded pilot in memory.
source_docs = {}
chunk_evidence = {}
for job in DATASETS:
    source_docs[job] = []
    chunk_evidence[job] = []
    for rec in selected[job]:
        path = locate_chunk(rec['name'])
        if path is None:
            raise FileNotFoundError(f'Chunk not found: {rec["name"]}. Edit SEARCH_ROOTS.')
        accepted = 0
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            for line in f:
                if len(source_docs[job]) >= DOCS_PER_DATASET: break
                if not line.strip(): continue
                obj = json.loads(line)
                value = obj.get(TEXT_FIELD)
                if not isinstance(value, str):
                    raise ValueError(f'Missing string field {TEXT_FIELD} in {path}')
                if not value.strip(): continue
                source_docs[job].append(value)
                accepted += 1
        chunk_evidence[job].append({'chunk':rec['name'],'documents_used':accepted})
        if len(source_docs[job]) >= DOCS_PER_DATASET: break
    assert len(source_docs[job]) == DOCS_PER_DATASET, f'{job}: only {len(source_docs[job])} docs'
    print(job, 'documents:', len(source_docs[job]))
assert sum(map(len,source_docs.values())) == 6000
print('STRATIFIED QUOTAS: PASS')

## 3. Content-hash split and EOS-aware tokenization

A normalized exact-content hash assigns identical documents to the same split across all six sources. A 5% hash threshold is **probabilistic**, so actual counts need not be exactly 50 validation documents per source. We stop if any source has no validation documents. Duplicate occurrences are tracked, and repeated exact documents are **skipped** in this pilot to avoid duplicate weighting. This does not detect near duplicates.

In [ ]:
def normalized_text(text):
    return re.sub(r'\s+', ' ', text).strip()

def digest_of(text):
    return hashlib.sha256(normalized_text(text).encode('utf-8')).hexdigest()

def split_for_digest(digest):
    return 'validation' if int(digest[:16], 16) % 100 < VALIDATION_PERCENT else 'train'

streams = {'train':[], 'validation':[]}
source_split_docs = Counter()
source_split_tokens = Counter()
seen_hashes = {}
duplicate_occurrences = 0
source_split_hashes = defaultdict(set)
doc_boundaries = {'train':[], 'validation':[]}
for job in DATASETS:
    for doc in source_docs[job]:
        digest = digest_of(doc)
        split = split_for_digest(digest)
        if digest in seen_hashes:
            duplicate_occurrences += 1
            assert seen_hashes[digest] == split
            continue  # Explicit exact deduplication policy for the pilot
        seen_hashes[digest] = split
        ids = tok.encode(doc, add_special_tokens=False, truncation=False)
        ids.append(tok.eos_token_id)
        start = len(streams[split])
        streams[split].extend(ids)
        doc_boundaries[split].append((start, len(streams[split]), job, digest))
        source_split_docs[(job,split)] += 1
        source_split_tokens[(job,split)] += len(ids)
        source_split_hashes[split].add(digest)

assert source_split_hashes['train'].isdisjoint(source_split_hashes['validation'])
assert all(source_split_docs[(job,'train')]>0 and source_split_docs[(job,'validation')]>0 for job in DATASETS), 'One source has an empty split; adjust SEED/VALIDATION_PERCENT and rerun in NEW output'
assert sum(source_split_docs.values()) + duplicate_occurrences == 6000
assert all(streams[p] for p in streams)
print('Exact duplicate occurrences skipped:', duplicate_occurrences)
for job in DATASETS:
    print(job, 'train:',source_split_docs[(job,'train')], 'validation:',source_split_docs[(job,'validation')])
print('SPLIT AND LEAKAGE CHECKS: PASS')

## 4. Packing and reconstruction invariants

Tokens are packed separately by split into complete blocks of 1,024. We verify that concatenating the packed blocks reconstructs the original token stream prefix exactly, and that the discarded tail is precisely the remainder. We also verify every document ends with EOS and that no EOS was inserted between sub-blocks of a document.

In [ ]:
blocks = {}
tails = {}
for split, stream in streams.items():
    nfull = len(stream)//BLOCK_SIZE
    blocks[split] = [stream[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE] for i in range(nfull)]
    tails[split] = stream[nfull*BLOCK_SIZE:]
    assert all(len(b)==BLOCK_SIZE for b in blocks[split])
    assert [x for b in blocks[split] for x in b] == stream[:nfull*BLOCK_SIZE]
    assert [x for b in blocks[split] for x in b] + tails[split] == stream
    assert len(tails[split]) == len(stream)%BLOCK_SIZE
    assert all(stream[end-1] == tok.eos_token_id for start,end,job,digest in doc_boundaries[split])
    print(split, 'documents:',sum(source_split_docs[(j,split)] for j in DATASETS),
          'tokens (including EOS):',len(stream),'full blocks:',len(blocks[split]),
          'tail tokens:',len(tails[split]))
assert len(blocks['validation'])>0 and len(blocks['train'])>0
print('PACKING / EOS / RECONSTRUCTION: PASS')

## 5. Persist the pilot report (no production shards)

Only evidence and statistics are written, not the complete token arrays. **No original files are modified.** Output directory must not exist before the run. The split is deterministic for identical inputs and settings, but this pilot is not a representative corpus quality audit.

In [ ]:
report = {
    'schema_version':'2.0',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'purpose':'stratified pretraining preparation pilot, not production data',
    'source_freeze_id':manifest['freeze_id'],
    'source_freeze_manifest_sha256':sha256_file(MANIFEST_PATH),
    'historical_tokenizer_identity_verified':False,
    'candidate_tokenizer_fingerprint_expected':EXPECTED_FINGERPRINT,
    'tokenizer_file_sha256':files,
    'seed':SEED, 'datasets':DATASETS, 'documents_per_dataset_requested':DOCS_PER_DATASET,
    'validation_percent_hash_threshold':VALIDATION_PERCENT,
    'block_size':BLOCK_SIZE, 'text_field':TEXT_FIELD,
    'tokenizer_eos_id':tok.eos_token_id,
    'source_chunk_selection':chunk_evidence,
    'documents_per_source_split':{job:{split:source_split_docs[(job,split)] for split in ('train','validation')} for job in DATASETS},
    'tokens_per_source_split':{job:{split:source_split_tokens[(job,split)] for split in ('train','validation')} for job in DATASETS},
    'duplicates_skipped':duplicate_occurrences,
    'totals':{split:{'documents':sum(source_split_docs[(j,split)] for j in DATASETS),
                     'tokens_including_eos':len(streams[split]),
                     'full_blocks':len(blocks[split]),
                     'tail_tokens':len(tails[split])} for split in ('train','validation')},
    'checks':{'per_source_quota':True,'both_splits_each_source':True,
              'exact_hash_split_disjoint':True,'eos_per_document':True,
              'packed_prefix_reconstructs':True,'tail_reconstructs':True},
    'environment':{'python':platform.python_version(), 'transformers':transformers.__version__, 'tokenizers':tokenizers.__version__},
    'limitations':['Historical tokenizer identity remains unproven.',
                   'Hash split prevents only exact normalized-text overlap in sampled documents.',
                   'Near-duplicate leakage, licenses, and full-corpus quality have not been assessed.',
                   'Pilot sampling is bounded and not a production distribution.',
                   'EOS adds one token per accepted document; token totals differ from raw audit.',
                   'Block size is provisional, pending architecture freeze.']
}
# Use exclusive creation to prevent accidental overwrite or reuse of prior pilot evidence.
OUTPUT.mkdir(parents=True, exist_ok=False)
report_path = OUTPUT/'pilot_report_v2.json'
with report_path.open('x', encoding='utf-8') as f:
    json.dump(report,f,indent=2,ensure_ascii=False)
report_sha = sha256_file(report_path)
(OUTPUT/'SHA256SUMS.txt').write_text(f'{report_sha}  pilot_report_v2.json\n',encoding='utf-8')
print('REPORT:',report_path)
print('SHA-256:',report_sha)
print('STATUS: STRATIFIED_PILOT_PASS')

## Interpretation and release gate

**Pass** means: 1,000 source documents were selected from each of six sources; both splits contain each source; exact normalized duplicates cannot cross splits; EOS and block reconstruction invariants passed. It does **not** establish historical tokenizer identity, production-ready splits, model architecture, or full-corpus near-deduplication.

**Next:** inspect `pilot_report_v2.json`, then design a resumable production builder with a globally consistent split/dedup strategy, shard manifests, and final architecture settings.